# RUNECLAW MAX — Colab Training Notebook

Fine-tune a trading AI model using the GetClaw Confluence Engine.

**Auto-detects GPU and selects best model:**
- T4 (16GB): Llama 3.2 3B
- L4/A10 (24GB): Llama 3.1 8B
- A100 (40/80GB): Llama 3.1 8B with rank-128 LoRA, or 70B

**Runtime → Change runtime type → GPU (A100 recommended)**

## 1. Install Dependencies

In [ ]:
%%capture
!pip install unsloth
!pip install --no-deps xformers trl peft accelerate bitsandbytes
!pip install safetensors sentencepiece protobuf

## 2. Detect GPU & Select Model

In [ ]:
import torch
import os

gpu_name = torch.cuda.get_device_name(0)
props = torch.cuda.get_device_properties(0)
vram_gb = props.total_memory / 1024**3

print(f"GPU:  {gpu_name}")
print(f"VRAM: {vram_gb:.1f} GB")
print(f"CUDA: {torch.version.cuda}")
print(f"PyTorch: {torch.__version__}")

# Auto-select model based on VRAM
if vram_gb >= 70:
    MODEL = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"
    LORA_RANK = 128
    BATCH_SIZE = 8
    MAX_SEQ = 2048
    print("\nConfig: 8B + rank-128 LoRA (A100 80GB mode)")
elif vram_gb >= 35:
    MODEL = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"
    LORA_RANK = 64
    BATCH_SIZE = 4
    MAX_SEQ = 2048
    print("\nConfig: 8B + rank-64 LoRA (A100 40GB mode)")
elif vram_gb >= 20:
    MODEL = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"
    LORA_RANK = 64
    BATCH_SIZE = 2
    MAX_SEQ = 2048
    print("\nConfig: 8B + rank-64 LoRA (L4/A10 mode)")
elif vram_gb >= 14:
    MODEL = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"
    LORA_RANK = 32
    BATCH_SIZE = 4
    MAX_SEQ = 1024
    print("\nConfig: 3B + rank-32 LoRA (T4 mode)")
else:
    MODEL = "unsloth/Llama-3.2-1B-Instruct-bnb-4bit"
    LORA_RANK = 16
    BATCH_SIZE = 4
    MAX_SEQ = 512
    print("\nConfig: 1B + rank-16 LoRA (low VRAM mode)")

# Common settings
GRAD_ACCUM = max(1, 16 // BATCH_SIZE)
NUM_EPOCHS = 3
LEARNING_RATE = 5e-5
MODEL_NAME = "runeclaw-colab"

print(f"Model:     {MODEL}")
print(f"LoRA:      rank={LORA_RANK}")
print(f"Batch:     {BATCH_SIZE} x {GRAD_ACCUM} = {BATCH_SIZE * GRAD_ACCUM} effective")
print(f"Context:   {MAX_SEQ}")
print(f"Epochs:    {NUM_EPOCHS}")

## 3. Generate Training Data (33K+ samples)

In [ ]:
import json
import random

random.seed(42)

SYSTEM_PROMPT = (
    "You are RUNECLAW, an AI trading analyst. You analyze cryptocurrency "
    "markets using the GetClaw Confluence Engine (12 weighted indicators), "
    "enforce strict risk management through 23 automated checks, and generate "
    "structured trade ideas. You never execute without human confirmation. "
    "Capital preservation above all."
)

PAIRS = [
    "BTC/USDT", "ETH/USDT", "SOL/USDT", "BNB/USDT", "XRP/USDT",
    "ADA/USDT", "AVAX/USDT", "DOGE/USDT", "DOT/USDT", "MATIC/USDT",
    "LINK/USDT", "UNI/USDT", "ATOM/USDT", "FIL/USDT", "APT/USDT",
    "ARB/USDT", "OP/USDT", "SUI/USDT", "SEI/USDT", "INJ/USDT",
    "TIA/USDT", "JUP/USDT", "WIF/USDT", "NEAR/USDT", "RENDER/USDT",
    "FET/USDT", "RUNE/USDT", "STX/USDT", "ICP/USDT", "PEPE/USDT",
]

INDICATORS = [
    "RSI", "MACD", "Bollinger Bands", "EMA Cross", "Volume Profile",
    "OBV", "Stochastic RSI", "ADX", "Ichimoku Cloud", "VWAP",
    "Fibonacci Retracement", "ATR",
]

RISK_CHECKS = [
    "max_position_size", "portfolio_heat", "correlation_limit",
    "drawdown_threshold", "volatility_filter", "liquidity_check",
    "spread_limit", "funding_rate", "open_interest_divergence",
    "whale_activity", "exchange_reserve", "news_sentiment",
    "time_of_day", "consecutive_losses", "win_rate_threshold",
    "max_leverage", "stop_loss_required", "take_profit_required",
    "risk_reward_minimum", "daily_loss_limit", "weekly_loss_limit",
    "margin_utilization", "circuit_breaker",
]

REGIMES = ["trending_up", "trending_down", "ranging", "volatile", "accumulation", "distribution"]
DIRECTIONS = ["long", "short"]

PRICES = {
    "BTC": 68000, "ETH": 3800, "SOL": 180, "BNB": 620, "XRP": 0.62,
    "ADA": 0.48, "AVAX": 38, "DOGE": 0.16, "DOT": 7.5, "MATIC": 0.72,
    "LINK": 18, "UNI": 12, "ATOM": 9.5, "FIL": 6.2, "APT": 9.8,
    "ARB": 1.2, "OP": 2.8, "SUI": 1.4, "SEI": 0.45, "INJ": 28,
    "TIA": 8.5, "JUP": 1.1, "WIF": 2.8, "PEPE": 0.000012, "RENDER": 8.5,
    "FET": 2.3, "NEAR": 7.2, "ICP": 14, "RUNE": 5.5, "STX": 2.1,
}

def rp(base, pct=0.1):
    raw = base * (1 + random.uniform(-pct, pct))
    if base < 0.01: return round(raw, 8)
    elif base < 1: return round(raw, 6)
    elif base < 100: return round(raw, 4)
    return round(raw, 2)

def rpct(lo, hi): return round(random.uniform(lo, hi), 2)
def gpp(pair): return PRICES.get(pair.split('/')[0], 10.0)

print("Generating training data...")

In [ ]:
all_samples = []

# --- 1. Trade Analysis (3000) ---
for _ in range(3000):
    pair = random.choice(PAIRS)
    tf = random.choice(["15m","1h","4h","1d"])
    d = random.choice(DIRECTIONS)
    base = gpp(pair); entry = rp(base, 0.02)
    if entry == 0: entry = base
    if d=="long":
        stop=round(entry*(1-random.uniform(0.01,0.04)),6)
        tp1=round(entry*(1+random.uniform(0.02,0.06)),6)
    else:
        stop=round(entry*(1+random.uniform(0.01,0.04)),6)
        tp1=round(entry*(1-random.uniform(0.02,0.06)),6)
    risk_pct=round(abs(entry-stop)/entry*100,2) if entry>0 else 1.0
    reward_pct=round(abs(tp1-entry)/entry*100,2) if entry>0 else 1.0
    rr=round(reward_pct/risk_pct,2) if risk_pct>0 else 0
    conf=rpct(0.45,0.95); ni=random.randint(6,12)
    bl=random.randint(3,ni); brl=ni-bl
    ai=random.sample(INDICATORS,ni)
    regime=random.choice(REGIMES)
    out=(f"Trade Idea: {pair}\nDirection: {d.upper()}\nTimeframe: {tf}\n"
         f"Market Regime: {regime}\n\nEntry: {entry}\nStop Loss: {stop} ({risk_pct}%)\n"
         f"Take Profit: {tp1} ({reward_pct}%)\nRisk:Reward: 1:{rr}\n\n"
         f"Confluence: {conf} ({ni} indicators)\n"
         f"Bullish ({bl}): {', '.join(ai[:bl])}\nBearish ({brl}): {', '.join(ai[bl:])}\n\n"
         f"Position Size: {rpct(1,5)}% of portfolio\n"
         f"Status: {'APPROVED' if conf>0.6 and rr>=1.5 else 'REQUIRES_REVIEW'}")
    all_samples.append({"instruction":f"Analyze {pair} on {tf} for trade setups.","input":"","output":out})
print(f"Trade Analysis: {3000}")

# --- 2. Multi-Timeframe (3000) ---
for _ in range(3000):
    pair=random.choice(PAIRS); base=gpp(pair)
    tfa={}
    for tf in ["15m","1h","4h","1d"]:
        tfa[tf]={"trend":random.choice(["bullish","bearish","neutral"]),"str":rpct(0.2,0.95)}
    trends=[v["trend"] for v in tfa.values()]
    bc=trends.count("bullish"); brc=trends.count("bearish")
    if bc>=3: align="STRONG BULLISH"; d="long"
    elif brc>=3: align="STRONG BEARISH"; d="short"
    elif bc==2: align="WEAK BULLISH"; d="long"
    elif brc==2: align="WEAK BEARISH"; d="short"
    else: align="MIXED"; d="none"
    lines=[f"Multi-Timeframe Analysis: {pair}\n"]
    for tf,data in tfa.items(): lines.append(f"  {tf}: {data['trend'].upper()} (strength: {data['str']})")
    lines.append(f"\nAlignment: {align} ({max(bc,brc)}/4)")
    if d!="none":
        lines.append(f"Recommendation: {d.upper()}")
        lines.append(f"Confirm on 15m for precise entry")
    else:
        lines.append(f"Recommendation: STAND ASIDE — conflicting signals")
    all_samples.append({"instruction":f"Multi-timeframe analysis on {pair}.","input":"","output":"\n".join(lines)})
print(f"Multi-Timeframe: {3000}")

# --- 3. No-Trade Rejections (3000) ---
reject_reasons = [
    "Low confluence ({}/12 indicators)", "R:R {:.1f}:1 below 1.5:1 minimum",
    "Portfolio heat {}% exceeds 6% max", "Consecutive losses: {} (max 3)",
    "Daily drawdown {}% exceeds 2% limit", "Spread {}% exceeds 0.1% max",
    "Funding rate {}% — crowded trade", "Volume ${}M below $10M threshold",
    "Circuit breaker active", "Conflicting timeframe signals",
    "News event in {} hours", "Correlation with {} too high ({}%)",
]
for _ in range(3000):
    pair=random.choice(PAIRS); d=random.choice(DIRECTIONS)
    reason=random.choice(reject_reasons)
    try: reason=reason.format(random.randint(2,5),rpct(0.5,1.4),rpct(6.1,12),
        random.randint(3,7),rpct(2.1,5),rpct(0.11,0.5),rpct(0.05,0.3),
        rpct(0.5,9.9),random.randint(1,12),random.choice(PAIRS),random.randint(75,98))
    except: pass
    nf=random.randint(1,5); fc=random.sample(RISK_CHECKS,nf)
    out=(f"Trade Scan: {pair} ({d.upper()})\nStatus: REJECTED\n\n"
         f"Reason: {reason}\n\nFailed checks ({nf}/{len(RISK_CHECKS)}):\n")
    for c in fc: out+=f"  FAIL: {c}\n"
    out+="\nAction: NO TRADE. Capital preservation above all."
    all_samples.append({"instruction":f"Scan {pair} for {d} setups.","input":"","output":out})
print(f"No-Trade: {3000}")

# --- 4. Confluence Breakdown (3000) ---
for _ in range(3000):
    pair=random.choice(PAIRS); tf=random.choice(["1h","4h","1d"])
    scores={}
    for ind in INDICATORS:
        sig=random.choice(["bullish","bearish","neutral"])
        scores[ind]={"sig":sig,"w":round(random.uniform(0.5,2.0),1),"c":rpct(0.3,0.99)}
    tw=sum(v["w"] for v in scores.values())
    bw=sum(v["w"] for v in scores.values() if v["sig"]=="bullish")
    brw=sum(v["w"] for v in scores.values() if v["sig"]=="bearish")
    conf=round(max(bw,brw)/tw,2) if tw>0 else 0
    lines=[f"GetClaw Confluence: {pair} ({tf})\n{'='*45}\n"]
    for ind,d in scores.items():
        icon="+" if d["sig"]=="bullish" else "-" if d["sig"]=="bearish" else "~"
        lines.append(f"  [{icon}] {ind}: {d['sig'].upper()} (w:{d['w']}, conf:{d['c']})")
    lines.extend([f"\nConfluence: {conf}",
        f"Signal: {'BULLISH' if bw>brw else 'BEARISH' if brw>bw else 'NEUTRAL'}",
        f"Actionable: {'YES' if conf>0.6 else 'NO'}"])
    all_samples.append({"instruction":f"GetClaw confluence for {pair} on {tf}.","input":"","output":"\n".join(lines)})
print(f"Confluence: {3000}")

# --- 5. Risk Edge Cases (2000) ---
for _ in range(2000):
    pair=random.choice(PAIRS)
    scenario=random.choice(["flash crash","max drawdown","correlation spike",
        "liquidation cascade","exchange downtime","funding spike","stablecoin depeg"])
    out=(f"CIRCUIT BREAKER: {scenario.upper()}\nPair: {pair}\n\n"
         f"Protocol:\n  1. HALT all new orders\n  2. Review open positions\n"
         f"  3. Tighten stops to breakeven\n  4. Reduce exposure by 50%\n"
         f"  5. Await human confirmation\n\n"
         f"Recovery: Wait for ATR normalization, resume at 25% size.\n"
         f"Capital preservation is non-negotiable.")
    all_samples.append({"instruction":f"Handle {scenario} for {pair}.","input":"","output":out})
print(f"Risk Edge Cases: {2000}")

# --- 6. Position Sizing (2000) ---
for _ in range(2000):
    pair=random.choice(PAIRS); base=gpp(pair)
    port=random.choice([1000,5000,10000,25000,50000,100000])
    rpt=rpct(0.5,2.0); entry=rp(base,0.02)
    if entry==0: entry=base
    sp=rpct(1.0,4.0); stop=round(entry*(1-sp/100),6)
    ra=round(port*rpt/100,2); pr=abs(entry-stop)
    pu=round(ra/pr,4) if pr>0 else 0; pv=round(pu*entry,2)
    pp=round(pv/port*100,2) if port>0 else 0
    wr=rpct(0.4,0.65); aw=rpct(2,6); al=rpct(1,3)
    kelly=round(wr-(1-wr)/(aw/al),4) if al>0 else 0
    out=(f"Position Sizing: {pair}\n{'='*40}\n\n"
         f"Portfolio: ${port:,}\nRisk/trade: {rpt}% (${ra})\n"
         f"Entry: {entry}\nStop: {stop} ({sp}%)\n\n"
         f"Size: {pu} units (${pv:,})\nPortfolio %: {pp}%\n\n"
         f"Kelly: {kelly*100:.1f}% (half-Kelly: {kelly*50:.1f}%)\n"
         f"Max allowed: 5%\n{'APPROVED' if pp<=5 else 'REDUCED to 5% cap'}")
    all_samples.append({"instruction":f"Position size for {pair} with ${port:,} portfolio.","input":"","output":out})
print(f"Position Sizing: {2000}")

# --- 7. Market Regime (2000) ---
regime_data={
    "trending_up":("Buy dips to EMA support","ADX>25, above EMA 20/50"),
    "trending_down":("Short rallies or stand aside","ADX>25, below EMA 20/50"),
    "ranging":("Mean reversion: buy support, sell resistance","ADX<20, BB squeeze"),
    "volatile":("Reduce exposure, widen stops","ATR spike, BB expansion"),
    "accumulation":("Scale into longs on volume","OBV rising, price flat"),
    "distribution":("Reduce longs, prepare shorts","OBV declining, price flat"),
}
for _ in range(2000):
    pair=random.choice(PAIRS); regime=random.choice(REGIMES)
    strat,inds=regime_data[regime]
    adx=rpct(10,60)
    out=(f"Market Regime: {pair}\n{'='*40}\n\n"
         f"Regime: {regime.upper().replace('_',' ')}\n"
         f"ADX: {adx} ({'trending' if adx>25 else 'ranging'})\n"
         f"Indicators: {inds}\n\nStrategy: {strat}\n"
         f"Persistence: {'HIGH' if adx>35 else 'MODERATE' if adx>20 else 'LOW'}")
    all_samples.append({"instruction":f"Classify market regime for {pair}.","input":"","output":out})
print(f"Market Regime: {2000}")

# --- 8. Correlation (2000) ---
for _ in range(2000):
    p1,p2,p3=random.sample(PAIRS,3)
    c12=random.randint(-30,98); c13=random.randint(-30,98); c23=random.randint(-30,98)
    hc=any(abs(c)>70 for c in [c12,c13,c23])
    out=(f"Correlation Analysis\n{'='*40}\n\n"
         f"  {p1} <-> {p2}: {c12}%\n  {p1} <-> {p3}: {c13}%\n  {p2} <-> {p3}: {c23}%\n\n")
    if hc: out+="WARNING: High correlation. Pick highest-conviction only.\nCombined risk max 3%."
    else: out+="Diversification: GOOD. Can hold simultaneous positions.\nMax portfolio heat: 6%."
    all_samples.append({"instruction":f"Correlation between {p1}, {p2}, {p3}.","input":"","output":out})
print(f"Correlation: {2000}")

# --- 9. Drawdown Recovery (1500) ---
for _ in range(1500):
    dd=rpct(1.5,15); nl=random.randint(2,10)
    port=random.choice([10000,25000,50000,100000]); da=round(port*dd/100,2)
    if dd<2: sev="NORMAL"; act="Continue with standard parameters."
    elif dd<5: sev="ELEVATED"; act="Reduce size 50%, raise confluence to 0.70, A+ setups only."
    elif dd<10: sev="HIGH"; act="HALT trading. Paper trade 20 signals. Full audit required."
    else: sev="CRITICAL"; act="EMERGENCY HALT. Close all. 1 week cooling off. Resume at 10% size."
    out=(f"DRAWDOWN PROTOCOL\n{'='*40}\n\n"
         f"Drawdown: {dd}% (${da:,})\nLosses: {nl} consecutive\n"
         f"Portfolio: ${port-da:,.2f} (from ${port:,})\n\n"
         f"Severity: {sev}\nAction: {act}")
    all_samples.append({"instruction":f"Handle {dd}% drawdown with {nl} consecutive losses.","input":"","output":out})
print(f"Drawdown: {1500}")

# --- 10. Chain of Thought (1500) ---
for _ in range(1500):
    pair=random.choice(PAIRS); d=random.choice(DIRECTIONS)
    conf=rpct(0.4,0.9); rr=rpct(0.8,4.0); heat=rpct(0,8); cl=random.randint(0,5)
    ok=conf>0.6 and rr>=1.5 and heat<6 and cl<3
    out=(f"Chain-of-Thought: {pair} ({d.upper()})\n{'='*45}\n\n"
         f"1. Confluence: {conf} {'PASS' if conf>0.6 else 'FAIL'}\n"
         f"2. R:R: 1:{rr} {'PASS' if rr>=1.5 else 'FAIL'}\n"
         f"3. Portfolio heat: {heat}% {'PASS' if heat<6 else 'FAIL'}\n"
         f"4. Consec losses: {cl} {'PASS' if cl<3 else 'FAIL'}\n\n"
         f"DECISION: {'APPROVED' if ok else 'REJECTED'}")
    if not ok: out+="\nCapital preservation above all."
    all_samples.append({"instruction":f"Reasoning for {d} on {pair}.","input":"","output":out})
print(f"Chain-of-Thought: {1500}")

random.shuffle(all_samples)
print(f"\nTotal synthetic samples: {len(all_samples)}")

In [ ]:
# Optional: Upload your existing training data from local machine
# This adds your real bot logs (10K samples) to the synthetic data

UPLOAD_EXISTING = True  # Set to False to skip

if UPLOAD_EXISTING:
    from google.colab import files
    print("Upload your combined_training.jsonl (from local training_data folder):")
    print("(Click Cancel to skip and use synthetic data only)")
    try:
        uploaded = files.upload()
        for fname, content in uploaded.items():
            existing = []
            for line in content.decode('utf-8').strip().split('\n'):
                if line.strip():
                    existing.append(json.loads(line))
            all_samples.extend(existing)
            random.shuffle(all_samples)
            print(f"Added {len(existing)} existing samples. Total: {len(all_samples)}")
    except Exception as e:
        print(f"Skipped upload. Using {len(all_samples)} synthetic samples.")

# Save to disk
DATA_FILE = "/content/training_data.jsonl"
with open(DATA_FILE, "w") as f:
    for s in all_samples:
        f.write(json.dumps(s, ensure_ascii=False) + "\n")
print(f"\nSaved {len(all_samples)} samples to {DATA_FILE}")

## 3b. Claude Knowledge Distillation (Optional)

Uses Claude API to generate expert-quality training samples via knowledge distillation.
This produces higher-quality data than synthetic generation alone — Claude reasons through
each trade scenario like an expert analyst.

**Requirements:** Anthropic API key with credits. Set `USE_CLAUDE = True` to enable.

In [ ]:
USE_CLAUDE = True  # Set to False to skip Claude distillation
CLAUDE_SAMPLES = 500  # Total samples across all categories (~70 per category)
CLAUDE_CONCURRENT = 3  # Parallel API calls (keep low to avoid rate limits)

if USE_CLAUDE:
    !pip install -q anthropic
    import getpass
    ANTHROPIC_API_KEY = getpass.getpass("Enter your Anthropic API key: ")
    print("Key set. Will generate Claude samples next.")

In [ ]:
if USE_CLAUDE:
    import anthropic
    import time
    from concurrent.futures import ThreadPoolExecutor, as_completed

    client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

    # Auto-detect working model
    CLAUDE_MODEL = None
    for candidate in ["claude-sonnet-4-6", "claude-haiku-4-5-20251001",
                       "claude-sonnet-4-20250514", "claude-3-5-sonnet-20241022",
                       "claude-3-5-haiku-20241022", "claude-3-haiku-20240307"]:
        try:
            client.messages.create(model=candidate, max_tokens=10,
                messages=[{"role":"user","content":"test"}])
            CLAUDE_MODEL = candidate
            print(f"Using model: {CLAUDE_MODEL}")
            break
        except Exception:
            continue
    if not CLAUDE_MODEL:
        print("ERROR: No working Claude model found. Skipping distillation.")
        USE_CLAUDE = False

if USE_CLAUDE:
    samples_per_cat = CLAUDE_SAMPLES // 7

    # 7 categories of expert prompts
    def make_prompts():
        prompts = []
        cats = {
            "trade_analysis": (
                "You are an expert crypto trader. Generate a detailed trade analysis for {pair} on the {tf} timeframe. "
                "Include: market regime classification, confluence score from 12 indicators (RSI, MACD, Bollinger Bands, "
                "EMA Cross, Volume Profile, OBV, Stochastic RSI, ADX, Ichimoku Cloud, VWAP, Fibonacci, ATR), "
                "exact entry/stop/TP levels with percentages, R:R ratio, position sizing recommendation, "
                "and risk check results (23 checks). Be specific with numbers. End with APPROVED or REJECTED."
            ),
            "multi_timeframe": (
                "Perform a multi-timeframe analysis for {pair} across 15m, 1h, 4h, and 1d timeframes. "
                "For each timeframe, describe the trend (bullish/bearish/neutral) with strength score. "
                "Assess alignment across timeframes. Give a final recommendation: LONG, SHORT, or STAND ASIDE. "
                "Explain your reasoning like an experienced trader would."
            ),
            "risk_management": (
                "A trader wants to {direction} {pair} with a ${portfolio} portfolio. Current market regime is {regime}. "
                "Walk through all 23 risk management checks: max_position_size, portfolio_heat, correlation_limit, "
                "drawdown_threshold, volatility_filter, liquidity_check, spread_limit, funding_rate, "
                "open_interest_divergence, whale_activity, exchange_reserve, news_sentiment, time_of_day, "
                "consecutive_losses, win_rate_threshold, max_leverage, stop_loss_required, take_profit_required, "
                "risk_reward_minimum, daily_loss_limit, weekly_loss_limit, margin_utilization, circuit_breaker. "
                "For each, state PASS or FAIL with reasoning. Give final verdict."
            ),
            "regime_adaptation": (
                "The market for {pair} has just transitioned from {regime1} to {regime2}. "
                "Explain: what indicators signaled this transition, how trading strategy should adapt, "
                "what position adjustments are needed, and what the optimal approach is for the new regime. "
                "Reference specific indicator values and thresholds."
            ),
            "no_trade": (
                "Analyze {pair} for a potential {direction} trade on {tf}. The setup looks tempting but should be "
                "REJECTED. Explain in detail why: identify which of the 23 risk checks fail, what the confluence "
                "score reveals, why the R:R is insufficient, or what market conditions make this dangerous. "
                "Emphasize capital preservation. Be specific about numbers and thresholds."
            ),
            "post_trade": (
                "Review a completed {direction} trade on {pair}. Entry was at a {quality} level. "
                "The trade {outcome}. Analyze: what went right/wrong, which indicators were accurate, "
                "how position sizing performed, what lessons apply to future trades. "
                "Include specific metrics and actionable improvements."
            ),
            "educational": (
                "Explain the concept of {topic} in the context of crypto trading with the GetClaw Confluence Engine. "
                "Cover: definition, how it's calculated, optimal thresholds, common mistakes, "
                "how it integrates with the other 11 indicators, and real examples with specific pairs and numbers. "
                "Target audience: intermediate trader."
            ),
        }

        for cat, template in cats.items():
            for i in range(samples_per_cat):
                pair = random.choice(PAIRS)
                tf = random.choice(["15m","1h","4h","1d"])
                direction = random.choice(["long","short"])
                portfolio = random.choice([5000,10000,25000,50000,100000])
                regime = random.choice(REGIMES)
                regime1, regime2 = random.sample(REGIMES, 2)
                quality = random.choice(["good","poor","excellent","mediocre"])
                outcome = random.choice(["hit TP1 at +3.2%","stopped out at -1.8%",
                    "hit TP2 at +5.5%","was manually closed at breakeven",
                    "hit full TP at +8%","stopped out at -2.5% on a wick"])
                topic = random.choice(["confluence scoring","position sizing with Kelly criterion",
                    "drawdown recovery protocols","multi-timeframe alignment",
                    "circuit breaker mechanisms","correlation-based portfolio limits",
                    "market regime detection using ADX","risk:reward optimization",
                    "volatility-adjusted stop losses","funding rate analysis"])

                prompt = template.format(
                    pair=pair, tf=tf, direction=direction, portfolio=portfolio,
                    regime=regime, regime1=regime1, regime2=regime2,
                    quality=quality, outcome=outcome, topic=topic
                )

                instruction_map = {
                    "trade_analysis": f"Analyze {pair} on {tf} for trade setups.",
                    "multi_timeframe": f"Multi-timeframe analysis on {pair}.",
                    "risk_management": f"Risk check for {direction} {pair} with ${portfolio:,} portfolio.",
                    "regime_adaptation": f"Market regime changed for {pair}. How to adapt?",
                    "no_trade": f"Scan {pair} for {direction} setups on {tf}.",
                    "post_trade": f"Review completed {direction} trade on {pair}.",
                    "educational": f"Explain {topic} for crypto trading.",
                }
                prompts.append({
                    "category": cat,
                    "prompt": prompt,
                    "instruction": instruction_map[cat],
                })
        random.shuffle(prompts)
        return prompts

    prompts = make_prompts()
    print(f"Generated {len(prompts)} Claude prompts across 7 categories")

    # Generate with concurrency and progress tracking
    claude_samples = []
    errors = 0

    def call_claude(item):
        try:
            resp = client.messages.create(
                model=CLAUDE_MODEL,
                max_tokens=1500,
                temperature=0.7,
                system=SYSTEM_PROMPT,
                messages=[{"role": "user", "content": item["prompt"]}],
            )
            text = resp.content[0].text.strip()
            if len(text) > 50:
                return {"instruction": item["instruction"], "input": "", "output": text}
        except Exception as e:
            if "rate" in str(e).lower():
                time.sleep(5)
            return None
        return None

    print(f"\nGenerating {len(prompts)} samples with {CLAUDE_CONCURRENT} threads...")
    with ThreadPoolExecutor(max_workers=CLAUDE_CONCURRENT) as executor:
        futures = {executor.submit(call_claude, p): p for p in prompts}
        done = 0
        for future in as_completed(futures):
            done += 1
            result = future.result()
            if result:
                claude_samples.append(result)
            else:
                errors += 1
            if done % 25 == 0 or done == len(prompts):
                print(f"  Progress: {done}/{len(prompts)} | Success: {len(claude_samples)} | Errors: {errors}")

    print(f"\nClaude distillation complete: {len(claude_samples)} expert samples")

    # Merge with existing data
    all_samples.extend(claude_samples)
    random.shuffle(all_samples)

    # Re-save combined data
    DATA_FILE = "/content/training_data.jsonl"
    with open(DATA_FILE, "w") as f:
        for s in all_samples:
            f.write(json.dumps(s, ensure_ascii=False) + "\n")
    print(f"Total training samples (synthetic + Claude): {len(all_samples)}")
    print(f"Saved to {DATA_FILE}")
else:
    if 'USE_CLAUDE' in dir():
        print("Claude distillation skipped. Using synthetic data only.")

## 4. Load Model & Apply LoRA

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL,
    max_seq_length=MAX_SEQ,
    dtype=None,
    load_in_4bit=True,
)
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha=LORA_RANK,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
model.print_trainable_parameters()

## 5. Tokenize & Train

In [ ]:
# Format and tokenize
formatted = []
for ex in all_samples:
    msgs = [{"role":"system","content":SYSTEM_PROMPT}]
    user = ex["instruction"]
    if ex.get("input","").strip(): user += "\n\n" + ex["input"]
    msgs.append({"role":"user","content":user})
    msgs.append({"role":"assistant","content":ex["output"]})
    formatted.append(tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False))

print(f"Formatted {len(formatted)} samples")

# Check lengths
lens = [len(tokenizer(t, truncation=False)["input_ids"]) for t in formatted[:500]]
print(f"Avg tokens: {sum(lens)/len(lens):.0f}, Max: {max(lens)}, Under {MAX_SEQ}: {sum(1 for l in lens if l<=MAX_SEQ)}/500")

In [ ]:
from torch.utils.data import Dataset as TorchDataset
from torch.nn.utils.rnn import pad_sequence

class DS(TorchDataset):
    def __init__(self, texts, tok, ml):
        self.items = []
        for i, t in enumerate(texts):
            enc = tok(t, truncation=True, max_length=ml, padding=False, return_tensors="pt")
            ids = enc["input_ids"].squeeze()
            self.items.append({"input_ids":ids,"attention_mask":torch.ones_like(ids),"labels":ids.clone()})
            if (i+1)%5000==0: print(f"  {i+1}/{len(texts)}...")
        print(f"  Tokenized {len(self.items)} samples")
    def __len__(self): return len(self.items)
    def __getitem__(self, i): return self.items[i]

class Collator:
    def __init__(self, pad): self.pad = pad
    def __call__(self, batch):
        return {
            "input_ids": pad_sequence([b["input_ids"] for b in batch], batch_first=True, padding_value=self.pad),
            "labels": pad_sequence([b["labels"] for b in batch], batch_first=True, padding_value=-100),
            "attention_mask": pad_sequence([b["attention_mask"] for b in batch], batch_first=True, padding_value=0),
        }

ds = DS(formatted, tokenizer, MAX_SEQ)
collator = Collator(tokenizer.pad_token_id)

In [ ]:
from transformers import Trainer, TrainingArguments

total_steps = len(ds) * NUM_EPOCHS // (BATCH_SIZE * GRAD_ACCUM)
warmup = int(total_steps * 0.05)

print(f"Training: {len(ds)} samples, {NUM_EPOCHS} epochs, {total_steps} steps")
print(f"Batch: {BATCH_SIZE} x {GRAD_ACCUM} = {BATCH_SIZE*GRAD_ACCUM}")
print(f"Warmup: {warmup} steps")

args = TrainingArguments(
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    warmup_steps=warmup,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=42,
    output_dir=f"/content/{MODEL_NAME}-checkpoints",
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
)

trainer = Trainer(model=model, args=args, train_dataset=ds, data_collator=collator)
stats = trainer.train()

print(f"\nDone! Loss: {stats.training_loss:.4f}, Runtime: {stats.metrics['train_runtime']:.0f}s")

## 6. Export to GGUF & Download

In [ ]:
# Export directly to GGUF using unsloth (works on Colab)
OUTPUT_DIR = f"/content/{MODEL_NAME}-gguf"

print("Exporting to GGUF (Q4_K_M)...")
model.save_pretrained_gguf(
    OUTPUT_DIR,
    tokenizer,
    quantization_method="q4_k_m",
)

# Find the GGUF file
import glob
gguf_files = glob.glob(f"{OUTPUT_DIR}/*.gguf")
for f in gguf_files:
    size = os.path.getsize(f) / 1024**3
    print(f"  {f}: {size:.1f} GB")

In [ ]:
# Create Modelfile for Ollama
gguf_name = os.path.basename(gguf_files[0]) if gguf_files else "unsloth.Q4_K_M.gguf"

modelfile = f"""FROM ./{gguf_name}

PARAMETER temperature 0.3
PARAMETER top_p 0.9
PARAMETER num_ctx 4096
PARAMETER stop "<|eot_id|>"
PARAMETER stop "<|end|>"

SYSTEM \"\"\"{SYSTEM_PROMPT}\"\"\"
"""

with open(f"{OUTPUT_DIR}/Modelfile", "w") as f:
    f.write(modelfile)

print(f"Modelfile created at {OUTPUT_DIR}/Modelfile")
print(f"\nFiles ready for download:")
for f in os.listdir(OUTPUT_DIR):
    size = os.path.getsize(f"{OUTPUT_DIR}/{f}") / 1024**2
    print(f"  {f}: {size:.1f} MB")

In [ ]:
# Option A: Download GGUF directly (if small enough)
from google.colab import files

gguf_path = gguf_files[0]
gguf_size = os.path.getsize(gguf_path) / 1024**3

if gguf_size < 4:  # Direct download if under 4GB
    print(f"Downloading {gguf_name} ({gguf_size:.1f} GB)...")
    files.download(gguf_path)
    files.download(f"{OUTPUT_DIR}/Modelfile")
else:
    print(f"GGUF is {gguf_size:.1f} GB — too large for direct download.")
    print("Use Option B (Google Drive) instead.")

In [ ]:
# Option B: Save to Google Drive (for large files)
from google.colab import drive
drive.mount('/content/drive')

import shutil
drive_dir = f"/content/drive/MyDrive/runeclaw-model"
os.makedirs(drive_dir, exist_ok=True)

for f in os.listdir(OUTPUT_DIR):
    src = f"{OUTPUT_DIR}/{f}"
    dst = f"{drive_dir}/{f}"
    print(f"Copying {f}...")
    shutil.copy2(src, dst)

print(f"\nSaved to Google Drive: {drive_dir}")
print(f"\nOn your local machine:")
print(f"  1. Download from Google Drive")
print(f"  2. cd to the folder")
print(f"  3. ollama create runeclaw -f Modelfile")
print(f"  4. ollama run runeclaw 'Scan BTC/USDT for trade setups'")

## 7. Quick Test (Optional)
Test the model before downloading.

In [ ]:
# Quick inference test
FastLanguageModel.for_inference(model)

test_prompt = tokenizer.apply_chat_template([
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "Scan BTC/USDT for trade setups on the 4h timeframe."},
], tokenize=False, add_generation_prompt=True)

inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=512, temperature=0.3, top_p=0.9)
response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print("=" * 60)
print("RUNECLAW Test Response:")
print("=" * 60)
print(response)